# LLM Lab 10: Alignment Experiment (DPO)
**Login:** run the first code cell. It prints a link and a short code. Open the link, enter the code and approve.
Colab then gets the Hugging Face token itself, so you don't create or paste any token.


In [ ]:
!pip install -q -U trl peft transformers datasets accelerate pandas huggingface_hub

# Connect to Hugging Face with no token typed or pasted.
# A link and a short code are shown below: open the link, enter the code, approve.
from huggingface_hub import login, whoami
login()

try:
    print("Logged in as:", whoami()["name"])
except Exception as e:
    print("Login is not finished yet. Complete the approval step above, then run this cell again.")


### Resolving Dependency Conflict
To resolve the dependency warning, we reinstall the required packages while keeping `pandas` pinned to the version compatible with Google Colab (`pandas==2.2.3`).

In [ ]:
# Re-installing with pinned pandas to fix the Google Colab compatibility conflict
!pip install -q -U trl peft transformers datasets accelerate "pandas==2.2.3"

### Resolving pyarrow/cudf Conflict
We can install a version of `pyarrow` that is compatible with `cudf-cu13` (less than 24, but at least 19.0.0) to clear the remaining dependency warning.

In [ ]:
# Re-upgrade pyarrow to satisfy Hugging Face 'datasets' requirements
# We prioritize 'datasets' over 'cudf' since we are running LLM / Hugging Face workflows
!pip install -q "pyarrow>=24.0.0"

Cell 2: inspect the preference dataset (requirement: "inspect chosen and rejected")

In [ ]:
from datasets import load_dataset
import numpy as np

ds = load_dataset("trl-lib/ultrafeedback_binarized")
print(ds)

for i in range(3):
    ex = ds["train"][i]
    print(f"\n===== EXAMPLE {i} =====")
    print("PROMPT  :", ex["chosen"][0]["content"][:300])
    print("\nCHOSEN  :", ex["chosen"][-1]["content"][:400])
    print("\nREJECTED:", ex["rejected"][-1]["content"][:400])

# simple evidence: length statistics
sub = ds["train"].select(range(1000))
cl = [len(x[-1]["content"].split()) for x in sub["chosen"]]
rl = [len(x[-1]["content"].split()) for x in sub["rejected"]]
print("\nAvg words chosen  :", np.mean(cl))
print("Avg words rejected:", np.mean(rl))

Cell 3: train DPO with two beta values

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

from huggingface_hub import whoami
HF_USER = whoami()["name"]      # your own Hugging Face username (from the login above)
print("Pushing models to account:", HF_USER)
BASE = "Qwen/Qwen2.5-0.5B"

train_ds = ds["train"].shuffle(seed=42).select(range(2000))
eval_ds  = ds["test"].select(range(100))

def train_dpo(beta):
    tok = AutoTokenizer.from_pretrained(BASE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # Load model in float16 to match fp16=True training on GPU
    model = AutoModelForCausalLM.from_pretrained(
        BASE,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    cfg = DPOConfig(
        output_dir=f"dpo-beta{beta}",
        beta=beta,
        learning_rate=5e-5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        max_length=512,
        fp16=True,
        logging_steps=10,
        eval_strategy="no",
        save_strategy="no",
        report_to="none",
    )
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules="all-linear", task_type="CAUSAL_LM")

    trainer = DPOTrainer(model=model, args=cfg, train_dataset=train_ds,
                         eval_dataset=eval_ds, processing_class=tok,
                         peft_config=lora)
    trainer.train()
    metrics = trainer.evaluate()
    print(f"\nbeta={beta} eval metrics:", metrics)

    merged = trainer.model.merge_and_unload().to(torch.float16)
    repo = f"{HF_USER}/qwen2.5-0.5b-dpo-beta{str(beta).replace('.','')}"
    merged.push_to_hub(repo)
    tok.push_to_hub(repo)
    log = trainer.state.log_history

    # Explicit clean up to prevent CUDA OOM on the subsequent run
    del trainer, model, merged
    gc.collect()
    torch.cuda.empty_cache()
    return repo, metrics, log

repo_01, m_01, log_01 = train_dpo(0.1)   # main run
repo_05, m_05, log_05 = train_dpo(0.5)   # additional beta experiment

Cell 4: test base vs aligned on the same prompts

In [ ]:
import pandas as pd

prompts = [
 "What is PPO (Proximal Policy Optimization) in reinforcement learning? Explain simply.",
 "What is the difference between PPO and DPO?",
 "I have a job interview in 2 days and I feel stressed. What should I do?",
 "Write a short, polite message to my class teacher asking for an extension on my assignment deadline.",
 "If a car travels 5 km in 2.5 hours, what is its speed? Show steps.",
 "Why did the Khan of Kalat give Gwadar to Oman in 1783?",
]

models = {
    "base": BASE,
    "dpo_beta_0.1": repo_01,
    "dpo_beta_0.5": repo_05,
}

def gen_all(model_id, prompts):
    tok = AutoTokenizer.from_pretrained(model_id)
    m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).cuda()
    stop = [tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>")]
    outs = []
    for p in prompts:
        text = tok.apply_chat_template([{"role": "user", "content": p}],
                                       tokenize=False, add_generation_prompt=True)
        inp = tok(text, return_tensors="pt").to("cuda")
        o = m.generate(**inp, max_new_tokens=200, do_sample=False,
                       repetition_penalty=1.1, eos_token_id=stop)
        outs.append(tok.decode(o[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip())
    del m; gc.collect(); torch.cuda.empty_cache()
    return outs

rows = {"prompt": prompts}
for name, mid in models.items():
    rows[name] = gen_all(mid, prompts)

df = pd.DataFrame(rows)
for c in ["base", "dpo_beta_0.1", "dpo_beta_0.5"]:
    df[f"{c}_score"] = ""    # fill manually 1-5
df.to_csv("evaluation.csv", index=False)
pd.set_option("display.max_colwidth", 400)
df[["prompt", "base", "dpo_beta_0.1", "dpo_beta_0.5"]]

In [ ]:
import pandas as pd

df = pd.read_csv("evaluation.csv")
cols = ["base_score", "dpo_beta_0.1_score", "dpo_beta_0.5_score"]
for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

summary = df[cols].mean().round(2)
summary.index = ["Base", "DPO beta=0.1", "DPO beta=0.5"]
print(summary)

# per-prompt table (useful as evidence in your report)
print(df[["prompt"] + cols].to_string(index=False))

summary.to_csv("summary_scores.csv")

In [ ]:
!pip install -q gradio
import gradio as gr, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from huggingface_hub import whoami
USER = whoami()["name"]
MODELS = {
    "base": "Qwen/Qwen2.5-0.5B",
    "dpo_0.1": f"{USER}/qwen2.5-0.5b-dpo-beta01",
    "dpo_0.5": f"{USER}/qwen2.5-0.5b-dpo-beta05",
}
_cache = {}

def load(key):
    if key not in _cache:
        tok = AutoTokenizer.from_pretrained(MODELS[key])
        model = AutoModelForCausalLM.from_pretrained(MODELS[key], torch_dtype=torch.float16).cuda().eval()
        _cache[key] = (tok, model)
    return _cache[key]

def generate(key, prompt, max_new_tokens, temperature):
    tok, model = load(key)
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to("cuda")
    stop = [tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>")]
    kw = dict(max_new_tokens=int(max_new_tokens), repetition_penalty=1.1, eos_token_id=stop)
    if temperature and temperature > 0:
        kw.update(do_sample=True, temperature=float(temperature), top_p=0.9)
    else:
        kw.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, **kw)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def before(prompt, mt, t):
    return generate("base", prompt, mt, t)

def after(prompt, beta, mt, t):
    return generate("dpo_0.1" if beta == "0.1" else "dpo_0.5", prompt, mt, t)

def compare(prompt, mt, t):
    return (generate("base", prompt, mt, t),
            generate("dpo_0.1", prompt, mt, t),
            generate("dpo_0.5", prompt, mt, t))

examples = prompts   # your six lab prompts

with gr.Blocks(title="Alignment Experiment: Qwen2.5-0.5B + DPO") as demo:
    gr.Markdown("# Alignment Experiment: Before vs After DPO\nBase: Qwen2.5-0.5B | Dataset: trl-lib/ultrafeedback_binarized")
    with gr.Tab("Before Alignment (Base)"):
        p1 = gr.Textbox(label="Prompt", lines=3)
        mt1 = gr.Slider(32, 300, 150, label="Max new tokens")
        t1 = gr.Slider(0, 1.2, 0, label="Temperature (0 = greedy)")
        o1 = gr.Textbox(label="Base model response", lines=10)
        gr.Button("Generate").click(before, [p1, mt1, t1], o1, api_name="before")
        gr.Examples(examples, p1)
    with gr.Tab("After Alignment (DPO)"):
        p2 = gr.Textbox(label="Prompt", lines=3)
        b2 = gr.Radio(["0.1", "0.5"], value="0.1", label="Beta value")
        mt2 = gr.Slider(32, 300, 150, label="Max new tokens")
        t2 = gr.Slider(0, 1.2, 0, label="Temperature (0 = greedy)")
        o2 = gr.Textbox(label="Aligned model response", lines=10)
        gr.Button("Generate").click(after, [p2, b2, mt2, t2], o2, api_name="after")
        gr.Examples(examples, p2)
    with gr.Tab("Compare Side by Side"):
        p3 = gr.Textbox(label="Prompt", lines=3)
        mt3 = gr.Slider(32, 300, 150, label="Max new tokens")
        t3 = gr.Slider(0, 1.2, 0, label="Temperature")
        with gr.Row():
            a = gr.Textbox(label="Base", lines=12)
            b = gr.Textbox(label="DPO beta=0.1", lines=12)
            c = gr.Textbox(label="DPO beta=0.5", lines=12)
        gr.Button("Compare").click(compare, [p3, mt3, t3], [a, b, c], api_name="compare")

demo.queue().launch(share=True, debug=True)